In [1]:
import pandas as pd

In [2]:
DATA_PATH = "../Data/Processed/"

In [3]:
customers = pd.read_csv(
    DATA_PATH + "customers.csv",
    dtype={"customer_zip_code_prefix": str}
)

geolocation = pd.read_csv(
    DATA_PATH + "geolocation.csv"
)

orders = pd.read_csv(
    DATA_PATH + "orders.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

order_items = pd.read_csv(
    DATA_PATH + "order_items.csv",
    parse_dates=["shipping_limit_date"]
)

order_payments = pd.read_csv(
    DATA_PATH + "order_payments.csv"
)

order_reviews = pd.read_csv(
    DATA_PATH + "order_reviews.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

products = pd.read_csv(
    DATA_PATH + "products.csv"
)

product_category_translation = pd.read_csv(
    DATA_PATH + "product_category_name_translation.csv"
)

sellers = pd.read_csv(
    DATA_PATH + "sellers.csv",
    dtype={"seller_zip_code_prefix": str}
)

In [4]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "product_category_translation": product_category_translation,
    "sellers": sellers
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (738332, 5)
orders: (99441, 8)
order_items: (112650, 8)
order_payments: (103886, 5)
order_reviews: (99224, 7)
products: (32951, 9)
product_category_translation: (71, 2)
sellers: (3095, 4)


In [5]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [6]:
order_items.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
item_total                    float64
dtype: object

In [7]:
customers.dtypes

customer_id                 str
customer_unique_id          str
customer_zip_code_prefix    str
customer_city               str
customer_state              str
dtype: object

In [8]:
sellers.dtypes

seller_id                 str
seller_zip_code_prefix    str
seller_city               str
seller_state              str
dtype: object

                 ORDER-LEVEL TABLES
            ┌──────────┬─────────────┐
            │          │             │
         orders     payments      reviews
            │          │             │
            └──────────┴─────────────┘
                       │
                aggregate where needed
                       │
                       ▼
                 ORDER_ITEMS
              ⭐ MAIN GRAIN ⭐
                       │
                 ┌─────┴─────┐
                 ▼           ▼
             products      sellers
                 │
                 ▼
              category

In [9]:
order_items[["order_id", "order_item_id"]].duplicated().sum()

np.int64(0)

In [10]:
order_items.groupby("order_id").size().describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

In [11]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [12]:
order_items["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    112650
Name: count, dtype: int64

## Building the Item-Level Analytical Dataset

In [13]:
item_analysis = order_items.copy()

In [14]:
item_analysis.shape

(112650, 8)

In [15]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [16]:
item_analysis = item_analysis.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ],
    on="order_id",
    how="left"
)

In [17]:
item_analysis.shape

(112650, 15)

In [18]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [19]:
item_analysis = item_analysis.merge(
    products[
        [
            "product_id",
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)

In [20]:
item_analysis.shape

(112650, 23)

In [21]:
sellers.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='str')

In [22]:
item_analysis = item_analysis.merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

In [23]:
item_analysis.shape

(112650, 26)

In [24]:
order_payments.groupby("order_id").size().describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

In [25]:
order_payments.groupby("order_id")["payment_type"].nunique().value_counts().sort_index()

payment_type
1    97194
2     2246
Name: count, dtype: int64

In [26]:
order_payments[
    order_payments.groupby("order_id")["payment_type"].transform("nunique") > 1
].head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,1,45.17
35,b2bb080b6bc860118a246fd9b6fad6da,1,credit_card,1,173.84
75,3689194c14ad4e2e7361ebd1df0e77b0,2,voucher,1,57.53
84,723e462ce1ee50e024887c0b403130f3,1,credit_card,1,13.80
102,21b8b46679ea6482cbf911d960490048,2,voucher,1,43.12
121,ea9184ad433a404df1d72fa0a8764232,4,voucher,1,22.49
161,c61e3efc183db6b89f5556efb5aa8797,1,credit_card,1,45.50
164,487c1451b8fd7347d0e80e5aca887e91,2,voucher,1,30.00
166,65c863d90026e9dd4888e28d2135c983,2,voucher,1,50.00
182,79b63d1f9cf0257b33b04fbfb0847cba,2,voucher,1,76.90


In [27]:
order_payments[
    order_payments["order_id"] == "5cfd514482e22bc992e7693f0e3e8df7"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,1,45.17
57742,5cfd514482e22bc992e7693f0e3e8df7,1,credit_card,4,665.41


In [28]:
(
    order_payments.groupby("order_id")["payment_sequential"].max()
    != order_payments.groupby("order_id").size()
).sum()

np.int64(80)

In [29]:
payment_check = (
    order_payments.groupby("order_id")
    .agg(
        max_sequence=("payment_sequential", "max"),
        payment_rows=("payment_sequential", "size")
    )
    .reset_index()
)

payment_check[
    payment_check["max_sequence"] != payment_check["payment_rows"]
].head(10)

,order_id,max_sequence,payment_rows
242,00ac05fe0fc047c54418098eb64e3aaa,2,1
2133,056c68d093c100017aab1f00f260705c,2,1
2518,0668d086b3ae41adde3aed3dacbc8fae,2,1
4111,0a7d898c6305101e69e9f5a05cd0130d,2,1
5123,0d0e88a418636d40ea780339701db49c,2,1
6196,0fdb6f73fab7005c84d4a43d8a93682b,2,1
8396,159da9914b51de617fe80162939965c5,2,1
8675,166b01d0fce75a595b92ad9e84e0843b,2,1
9538,1896e8b8fd196151559b02e31e98ce89,2,1
10244,1a57108394169c0b47d8f876acc9ba2d,2,1


In [30]:
payment_summary = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_types=("payment_type", lambda x: ", ".join(sorted(x.unique()))),
        max_payment_installments=("payment_installments", "max"),
        payment_count=("payment_value", "count")
    )
    .reset_index()
)

In [31]:
payment_summary.shape

(99440, 5)

In [32]:
payment_summary.head()

,order_id,total_payment_value,payment_types,max_payment_installments,payment_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,credit_card,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,credit_card,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,credit_card,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,credit_card,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,credit_card,3,1


In [33]:
payment_summary[
    payment_summary["order_id"] == "5cfd514482e22bc992e7693f0e3e8df7"
]

,order_id,total_payment_value,payment_types,max_payment_installments,payment_count
36023,5cfd514482e22bc992e7693f0e3e8df7,710.58,"credit_card, voucher",4,2


In [34]:
item_analysis = item_analysis.merge(
    payment_summary,
    on="order_id",
    how="left"
)

In [35]:
item_analysis.shape

(112650, 30)

In [36]:
order_reviews.groupby("order_id").size().describe()

count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
dtype: float64

In [37]:
order_reviews[
    order_reviews.groupby("order_id").transform("size") > 1
].sort_values("order_id").head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21,2018-03-22 01:32:08
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19,2017-07-20 12:06:11
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14,2017-07-17 13:58:06


In [38]:
order_reviews["review_id"].duplicated().sum()

np.int64(814)

In [39]:
#814 → duplicate occurrences beyond the first .duplicated().sum() → 3 (A's 2nd + 3rd, B's 2nd)
#1603 → all rows involved in duplicated review_ids .duplicated(keep=False) → 5 rows (A A A B B)

order_reviews[order_reviews["review_id"].duplicated(keep=False)].shape

(1603, 7)

In [40]:
order_reviews[
    order_reviews["review_id"].duplicated(keep=False)
].sort_values("review_id").head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44


In [41]:
order_reviews.groupby("order_id")["review_score"].nunique().value_counts().sort_index()

review_score
1    98471
2      202
Name: count, dtype: int64

In [42]:
order_reviews["review_score"].isna().sum()

np.int64(0)

In [43]:
review_summary = (
    order_reviews
    .groupby("order_id")
    .agg(
        review_count=("review_id", "count"),
        review_score_avg=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_score_max=("review_score", "max")
    )
    .reset_index()
)

In [44]:
review_summary.shape

(98673, 5)

In [45]:
review_summary.head()

,order_id,review_count,review_score_avg,review_score_min,review_score_max
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0,5,5
1,00018f77f2f0320c557190d7a144bdd3,1,4.0,4,4
2,000229ec398224ef6ca0657da4fc703e,1,5.0,5,5
3,00024acbcdf0a6daa1e931b038114c75,1,4.0,4,4
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0,5,5


In [46]:
review_summary[
    review_summary["review_count"] > 1
].head(10)

,order_id,review_count,review_score_avg,review_score_min,review_score_max
84,0035246a40f520710769010f752e7507,2,5.000000,5,5
461,013056cfe49763c6f66bda03396c5ee3,2,4.500000,4,5
556,0176a6846bcb3b0d3aa3116a9a768597,2,5.000000,5,5
835,02355020fd0a40a0d56df9f6ff060413,2,2.000000,1,3
985,029863af4b968de1e5d6a82782e662f5,2,4.500000,4,5
1092,02e0b68852217f5715fb9cc885829454,2,4.000000,4,4
1103,02e723e8edb4a123d414f56cc9c4665e,2,5.000000,5,5
1272,03515a836bb855b03f7df9dee520a8fc,2,5.000000,5,5
1455,03c939fd7fd3b38f8485a0f95798f1f6,3,3.333333,3,4
1510,03eba6d9fef8f5b3e811d4b5a7cca9cd,2,4.500000,4,5


In [47]:
item_analysis = item_analysis.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [48]:
item_analysis.shape

(112650, 34)

In [49]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [50]:
item_analysis = item_analysis.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

In [51]:
item_analysis.shape

(112650, 38)

In [52]:
product_category_translation["product_category_name"].duplicated().sum()

np.int64(0)

In [53]:
item_analysis = item_analysis.merge(
    product_category_translation[
        [
            "product_category_name",
            "product_category_name_english"
        ]
    ],
    on="product_category_name",
    how="left"
)

In [54]:
item_analysis.shape

(112650, 39)

In [55]:
item_analysis[["order_id", "order_item_id"]].duplicated().sum()

np.int64(0)